# 00 · Course Overview and Setup

**Part I — Foundations** · Estimated time: ~45 min

Welcome to *Deep Learning with PyTorch: From Foundations to Frontier*. This first notebook is a map, a
handshake with your computer and a first taste of the whole subject in one page. By the end of it you
will know what the course covers, you will have verified that your environment can run everything
that follows, and you will have trained your first neural network and looked at what it learned.

## Learning objectives

After working through this notebook you will be able to:

- Explain what deep learning is, how it relates to machine learning and artificial intelligence, and where its main ideas came from.
- Describe the structure of this course, its seven parts and eighteen notebooks, and choose a study schedule that fits your time.
- Verify your Python and PyTorch installation, detect a GPU, and interpret a CPU-versus-GPU timing experiment.
- Name the three ingredients of every deep learning system (model, loss, optimizer) and describe the training loop that ties them together.
- Train a small multi-layer perceptron on a two-dimensional classification problem and read a decision-boundary plot.
- Read a PyTorch error message and diagnose the most common environment problems.
- Use the notation and shape conventions that the rest of the course relies on.

## Prerequisites

None within the course. You should be comfortable with Python (functions, classes, lists and
dictionaries) and have seen NumPy arrays before. Everything else, including the mathematics, is taught
from Notebook 01 · Mathematical Foundations onwards.

## Contents

1. What is deep learning?
2. The course map
3. Environment check
4. Deep learning in one page
5. How to study this course
6. Notation and conventions

Exercises · Summary · Further reading

In [ ]:
import math, os, random, time, sys, platform, importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

# FAST_RUN = True shrinks every training budget so the whole notebook runs in a few
# minutes on a CPU. Set it to False for the full experiments (recommended on a GPU).
FAST_RUN = False
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

The cell above is the **setup cell**. Every notebook in this course starts with one that has exactly
this shape, so it is worth reading it once carefully.

- The imports bring in the standard library, NumPy, matplotlib and the three PyTorch namespaces we use
  constantly: `torch` (tensors), `torch.nn` (layers and losses) and `torch.nn.functional` (the same
  operations as plain functions).
- `SEED = 42` and the three `seed(...)` calls make the random numbers reproducible. Deep learning is
  full of randomness (initial weights, data shuffling, dropout), and being able to re-run an
  experiment and get the same numbers is the difference between science and anecdote.
- `device` is where tensors and models will live. If a CUDA-capable GPU is visible, we use it;
  otherwise we fall back to the CPU. All the code in this course is written to work on either.
- `FAST_RUN` is a switch that shrinks every experiment to a size that finishes in a few minutes on a
  laptop CPU. Section 2 explains the convention in detail.
- `DATA_DIR` points to the shared `data/` folder next to the `notebooks/` folder, where datasets are
  downloaded once and reused by every notebook.

## 1. What is deep learning?

**Machine learning** is the discipline of writing programs whose behaviour is determined by data rather
than by hand-written rules. Instead of telling the computer how to recognise a cat, we show it many
labelled pictures and let an algorithm adjust the parameters of a flexible function until it maps
pictures to the right labels. **Artificial intelligence** is the broader ambition of building systems
that behave intelligently; machine learning is currently the most successful route towards it.
**Deep learning** is the sub-field of machine learning in which the flexible function is a
**neural network** with many layers, and in which the *features* the network uses to make its
decisions are themselves learned from data rather than designed by an engineer.

That last point is the heart of the matter. In classical machine learning a practitioner spends most
of their time on **feature engineering**: deciding that the relevant properties of an e-mail are the
counts of certain words, or that the relevant properties of an image are the responses of certain
edge filters, and then feeding those hand-crafted numbers to a comparatively simple model such as
logistic regression. Deep learning replaces the hand-crafted pipeline with a stack of simple,
differentiable transformations, each of which produces a slightly more abstract description of the
input than the one before. Early layers of an image network learn edge and colour detectors, middle
layers learn textures and object parts, late layers learn whole objects. Nobody designs these
detectors; they emerge because they are useful for the final task. This is why deep learning is
often described as **representation learning**: the model learns not only the decision rule but the
representation on which the rule operates.

Three properties make a problem a good fit for deep learning. First, there must be **enough data**:
neural networks have millions or billions of adjustable parameters and need many examples to pin
them down, although Notebook 07 · Transfer Learning and Modern Vision shows how pre-trained networks
let us get away with far less. Second, the input should have **rich internal structure** that a
composition of layers can exploit: images, audio, text, molecules, game states and time series all
qualify, whereas a table of twenty unrelated numeric columns often does not (gradient-boosted trees
frequently win there). Third, there must be **compute**: the training procedure evaluates the
network and its derivatives millions of times, which is why the field took off only when graphics
processors made this affordable.

It is equally important to know when deep learning is the wrong tool. When data is scarce and
tabular, simpler models are usually better and far easier to interpret. When a decision must come
with a guarantee, a proof or a legal justification, a black-box model is a liability. When the
question is causal ("what would happen if we changed the price?") rather than predictive, no amount
of pattern matching on observational data answers it. And when the data the model sees in production
differ from the data it was trained on, deep networks can fail silently and confidently; Notebook 16 ·
Interpretability, Robustness and Uncertainty is devoted to detecting and mitigating this.

### 1.1 A short history

The ideas in this course are older than they look, and knowing where they came from helps to
understand why they have the form they have.

In **1958** Frank Rosenblatt introduced the **perceptron**, a single artificial neuron that computes a
weighted sum of its inputs and fires when the sum exceeds a threshold, together with a learning rule
that adjusts the weights from examples. It was hailed as the seed of machine intelligence, but in 1969
Minsky and Papert showed that a single perceptron cannot represent the exclusive-or function, and
interest collapsed for over a decade. We reproduce both the perceptron and its failure on XOR in
Notebook 02 · Neural Networks from Scratch (NumPy).

The cure was known in principle, namely stacking neurons in layers, but nobody had a practical way to
train the hidden layers. In **1986** Rumelhart, Hinton and Williams popularised **backpropagation**,
an efficient application of the chain rule that computes the derivative of the loss with respect to
every weight in a multi-layer network in one backward sweep. Backpropagation is the engine of the
entire field, and you will derive it by hand in Notebook 02 and build an automatic version of it in
Notebook 03 · Tensors and Autograd.

Through the 1990s neural networks were one method among many. Yann LeCun's **LeNet** (**1998**)
demonstrated that a network with **convolutional** layers, which share weights across positions in an
image, could read handwritten digits well enough to process a sizeable share of the cheques in the
United States. Hochreiter and Schmidhuber's **LSTM** (1997) addressed the problem of remembering
information over long sequences. Yet in the 2000s support vector machines and boosted trees were more
fashionable; networks were considered hard to train and poorly understood.

Everything changed in **2012**. Krizhevsky, Sutskever and Hinton trained **AlexNet**, an
eight-layer convolutional network, on two consumer GPUs and won the ImageNet competition by a margin
so large that the whole computer-vision community switched methods within two years. Most of the
ingredients were not new: convolutions, ReLU activations and stochastic gradient descent had all been
around for years (dropout, proposed by Hinton's group the same year, was the one recent addition).
What was new was the combination of a large labelled dataset, sufficient compute and the engineering
to use it.
In **2015** He et al. introduced the **residual connection**, which lets gradients flow through
skip paths and made it possible to train networks with more than a hundred layers (**ResNet**);
the same year saw batch normalisation and the Adam optimizer become standard. These are the topics
of Notebooks 04 to 07.

In **2017** Vaswani et al. published *Attention Is All You Need*, replacing recurrence with
**self-attention** and introducing the **Transformer**. Because attention is trivially parallel,
Transformers scale to data and model sizes that recurrent networks could never reach. **2018**
brought **BERT** and the first **GPT**, which showed that a Transformer pre-trained on raw text and
then fine-tuned dominated nearly every language benchmark. Around **2020** two further pieces fell
into place: **scaling laws** (Kaplan et al.) quantified how loss improves predictably with model
size, data and compute, and **denoising diffusion models** (Ho et al.) became a stable, high-quality
way to generate images. GPT-3 demonstrated that a sufficiently large language model can perform new
tasks from a handful of examples in its prompt. The years since have been the **LLM era**: chat
assistants built on instruction-tuned language models, text-to-image systems built on latent
diffusion, and the same Transformer architecture applied to vision, audio, protein structure and
robotics. Notebooks 09 to 12 cover these developments, and Notebooks 14 and 15 cover the engineering
that makes them affordable.

The cell below draws this timeline. It is our first use of matplotlib in the course, so it also
serves as a check that plotting works.

In [ ]:
milestones = [
    (1958, "Perceptron\n(Rosenblatt)"),
    (1986, "Backpropagation\n(Rumelhart et al.)"),
    (1998, "LeNet-5\n(LeCun et al.)"),
    (2012, "AlexNet\n(Krizhevsky et al.)"),
    (2015, "ResNet, BatchNorm,\nAdam"),
    (2017, "Transformer\n(Vaswani et al.)"),
    (2018, "BERT / GPT"),
    (2020, "Diffusion models,\nscaling laws, GPT-3"),
    (2023, "LLM era"),
]

fig, ax = plt.subplots(figsize=(10, 3.2))
years = [y for y, _ in milestones]
ax.hlines(0, 1955, 2026, color="gray", linewidth=1)
ax.scatter(years, [0] * len(years), s=60, zorder=3, color="C3")
for i, (year, label) in enumerate(milestones):
    offset = 0.6 if i % 2 == 0 else -0.6            # alternate labels above and below the line
    ax.annotate(f"{year}\n{label}", (year, 0), (year, offset), ha="center",
                va="bottom" if offset > 0 else "top", fontsize=8,
                arrowprops=dict(arrowstyle="-", color="gray", lw=0.8))
ax.set_ylim(-2.2, 2.2); ax.set_xlim(1955, 2027)
ax.set_yticks([]); ax.grid(False)
ax.set_xlabel("year"); ax.set_title("A compressed history of deep learning")
for side in ["left", "right", "top"]:
    ax.spines[side].set_visible(False)
plt.tight_layout(); plt.show()

Notice how the milestones cluster: nearly three decades of quiet between 1958 and 1986, another twelve
years until LeNet, and then, after 2012, something important almost every year. The acceleration is not a
coincidence. Each of the recent advances lowered the cost of the next one: GPUs made large networks
trainable, large networks made pre-training worthwhile, pre-training made Transformers dominant, and
Transformers made scaling laws visible. The course follows roughly the same order, which is also the
order in which the ideas depend on one another.

> **Key idea.** Deep learning is machine learning with learned representations: a composition of
> simple differentiable layers, trained end-to-end by gradient descent on large datasets.

## 2. The course map

The course has eighteen notebooks organised in seven parts. Each notebook is self-contained in the
sense that it re-defines the small helper functions it needs, so you can open any one of them
directly, but the concepts build on one another and the intended order is numerical. The table
below is the map you will want to come back to.

| # | Notebook | Part | ~min | What you will learn |
|---|----------|------|-----:|---------------------|
| 00 | Course Overview and Setup | I — Foundations | 45 | The map, the environment, one complete tiny example. |
| 01 | Mathematical Foundations | I — Foundations | 120 | Linear algebra, calculus, probability and numerics with NumPy code for every concept. |
| 02 | Neural Networks from Scratch (NumPy) | I — Foundations | 120 | Multi-layer perceptrons and backpropagation derived and implemented by hand; MNIST in NumPy. |
| 03 | Tensors and Autograd | I — Foundations | 120 | PyTorch tensors, a micrograd-style autograd engine, `torch.autograd` in depth. |
| 04 | Training Deep Networks: Optimization, Initialization, Normalization, Regularization | II — Training deep networks | 150 | SGD, momentum, Adam, schedules, initialisation, BatchNorm, dropout, diagnosing training. |
| 05 | The PyTorch Workflow and Best Practices | II — Training deep networks | 120 | `nn.Module`, `Dataset`/`DataLoader`, a proper training loop, logging, checkpoints, reproducibility, debugging. |
| 06 | Convolutional Neural Networks | III — Computer vision | 150 | Convolutions from scratch, LeNet, VGG-style nets and ResNets on CIFAR-10, visualising what CNNs learn. |
| 07 | Transfer Learning and Modern Vision | III — Computer vision | 120 | Pre-trained models, fine-tuning, Grad-CAM, Vision Transformers, U-Net segmentation, self-supervision. |
| 08 | Sequence Models: RNNs, LSTMs and GRUs | IV — Sequences and language | 120 | Recurrence, backpropagation through time, gated cells, a character-level language model, forecasting. |
| 09 | Attention and Transformers | IV — Sequences and language | 150 | Attention from first principles, the Transformer block, a GPT trained from scratch, generation, scaling. |
| 10 | NLP with Pretrained Transformers (Hugging Face) | IV — Sequences and language | 120 | Tokenisation, word embeddings, fine-tuning BERT-style models, LoRA, how LLMs are trained today. |
| 11 | Autoencoders and Variational Autoencoders | V — Generative models | 120 | Representation learning without labels, the ELBO, VAEs and their latent spaces. |
| 12 | GANs and Diffusion Models | V — Generative models | 150 | Adversarial training, WGAN, denoising diffusion, DDIM, classifier-free guidance. |
| 13 | Reinforcement Learning | VI — Reinforcement learning | 120 | MDPs, Q-learning, REINFORCE, DQN, actor–critic and PPO on Gymnasium environments. |
| 14 | Efficient Training and Scaling | VII — Engineering, trust and practice | 120 | Profiling, mixed precision, memory tricks, `torch.compile`, Optuna, distributed training. |
| 15 | Deployment and Model Optimization | VII — Engineering, trust and practice | 120 | Export, ONNX Runtime, quantisation, pruning, distillation, serving and monitoring. |
| 16 | Interpretability, Robustness and Uncertainty | VII — Engineering, trust and practice | 120 | Attribution, adversarial examples, distribution shift, calibration, fairness. |
| 17 | Capstone Projects | VII — Engineering, trust and practice | 90 + project | A reusable project template and six full project specifications. |

The seven parts form a deliberate arc. **Part I** builds the mathematics and implements everything by
hand so that PyTorch never feels like magic. **Part II** teaches how to train networks reliably and
how to organise real PyTorch code. **Parts III and IV** cover the two great application domains,
vision and language, and the two architectural families that dominate them, convolutional networks and
Transformers. **Part V** turns from discriminating to generating. **Part VI** covers decision making
by trial and error. **Part VII** is about making models fast, deployable and trustworthy, and ends with
projects that put everything together.

### 2.1 Suggested schedules

The estimated times in the table are for a careful first pass, including running every cell and
reading the interpretation of every result, but excluding the exercises. Budget roughly the same time
again for the exercises. Two schedules that have worked well:

**Eight-week intensive** (about 8–10 hours per week):

| Week | Notebooks |
|------|-----------|
| 1 | 00, 01, 02 |
| 2 | 03, 04 |
| 3 | 05, 06 |
| 4 | 07, 08 |
| 5 | 09, 10 |
| 6 | 11, 12 |
| 7 | 13, 14 |
| 8 | 15, 16, 17 (choose a project and produce a plan) |

**Sixteen-week course** (about 4–5 hours per week): one notebook per week, with Notebooks 00 and 01
sharing week 1, Notebooks 02 to 15 in weeks 2 to 15, and Notebooks 16 and 17 sharing week 16; the
capstone project itself then continues beyond the sixteen weeks. The sixteen-week schedule is the one we recommend if you are studying alongside a job or other courses;
the material rewards time spent experimenting, and rushing through the mathematics in Part I makes
everything later harder.

The following cell prints the eight-week plan and totals the estimated time, using a small Python
list of the course map. It is a trivial computation, but it establishes a habit we will keep
throughout the course: whenever something can be computed, we compute it rather than assert it.

In [ ]:
course_map = [
    (0, "Course Overview and Setup", 45), (1, "Mathematical Foundations", 120),
    (2, "Neural Networks from Scratch (NumPy)", 120), (3, "Tensors and Autograd", 120),
    (4, "Training Deep Networks: Optimization, Initialization, Normalization, Regularization", 150),
    (5, "The PyTorch Workflow and Best Practices", 120), (6, "Convolutional Neural Networks", 150),
    (7, "Transfer Learning and Modern Vision", 120), (8, "Sequence Models: RNNs, LSTMs and GRUs", 120),
    (9, "Attention and Transformers", 150), (10, "NLP with Pretrained Transformers (Hugging Face)", 120),
    (11, "Autoencoders and Variational Autoencoders", 120), (12, "GANs and Diffusion Models", 150),
    (13, "Reinforcement Learning", 120), (14, "Efficient Training and Scaling", 120),
    (15, "Deployment and Model Optimization", 120), (16, "Interpretability, Robustness and Uncertainty", 120),
    (17, "Capstone Projects", 90),
]
eight_week_plan = [[0, 1, 2], [3, 4], [5, 6], [7, 8], [9, 10], [11, 12], [13, 14], [15, 16, 17]]

total_minutes = sum(minutes for _, _, minutes in course_map)
print(f"{len(course_map)} notebooks, {total_minutes / 60:.1f} hours of guided material (plus exercises)\n")
for week, ids in enumerate(eight_week_plan, start=1):
    minutes = sum(course_map[i][2] for i in ids)
    titles = ", ".join(f"{i:02d}" for i in ids)
    print(f"week {week}: notebooks {titles:<12} ~{minutes / 60:.1f} h reading + about the same for exercises")

About thirty-six hours of guided material, so roughly four and a half hours of reading per week on the
eight-week plan and about the same again for exercises; the weeks are close to evenly loaded, which
is why the plan pairs the long Notebooks 04, 06, 09 and 12 with shorter neighbours.

### 2.2 How exercises and solutions work

Every notebook ends with four to six exercises of increasing difficulty, from a warm-up to a
challenge. Each exercise is followed immediately by a fully worked solution: a markdown cell that
explains the approach and a code cell that runs. This is deliberate. Self-paced learners rarely have
someone to ask, and an exercise without a solution is only useful until you get stuck. We nevertheless
recommend a simple discipline: **write your own attempt in a new cell before reading the solution**,
run it, and only then compare. If your solution differs from ours but works, that is fine; there are
usually several correct designs. If you could not get started, read the first paragraph of the
solution, close it, and try again.

### 2.3 The FAST_RUN convention

Deep learning experiments take time, and a course that requires a GPU for every cell would exclude
many learners. Every notebook therefore defines a boolean `FAST_RUN` in its setup cell, and **every
training loop reads its budget from it**, for example

```python
EPOCHS = 1 if FAST_RUN else 8
N_TRAIN = 4_000 if FAST_RUN else 60_000
```

With `FAST_RUN = True` the whole notebook runs in a few minutes on a laptop CPU, at the cost of
somewhat worse numbers (a model trained for one epoch on four thousand images is not as good as one
trained for eight epochs on sixty thousand). With `FAST_RUN = False`, which is the default in the
files, you get the full experiments; on a mid-range GPU a full notebook takes five to twenty minutes.
Our advice: on a first reading set `FAST_RUN = True` so you can move quickly and focus on
understanding, then come back to the experiments that interest you with the full budget. Whenever the
prose reports a number such as "about 97% test accuracy", it refers to the full budget.

### 2.4 The data folder

Datasets are downloaded into `../data`, one directory above the `notebooks/` folder, which the setup
cell exposes as `DATA_DIR`. The torchvision datasets (MNIST, FashionMNIST, CIFAR-10) download
themselves the first time they are used, with `download=True`, and are then cached; the total, including
the archives torchvision keeps next to the extracted files, is about half a gigabyte. Any other download in the course is wrapped in a `try/except` with a fallback, so that every
notebook still runs offline, if occasionally with a smaller built-in substitute. Notebook 10 · NLP
with Pretrained Transformers (Hugging Face) additionally caches pre-trained models in the Hugging
Face cache directory in your home folder.

### 2.5 Hardware expectations

Everything in the course runs on a CPU with 8 GB of RAM. A CUDA-capable NVIDIA GPU with at least 4 GB
of memory makes the full experiments in Parts III to VII pleasant rather than patient; nothing in the
course needs more than about 2 GB of GPU memory. Apple Silicon users can run everything on the CPU;
the `mps` backend works for most notebooks but is not what the course was tested on, so we keep the
device logic to `cuda` or `cpu`. If you have no GPU at all, free hosted notebooks (Google Colab, Kaggle
Notebooks) provide one; the notebooks run there unchanged apart from the data path.

## 3. Environment check

Before doing anything else, let us confirm that the environment is what we expect. The course was
developed against the `dl-course` conda environment described in `environment.yml` (Python 3.12,
PyTorch 2.x with CUDA, torchvision, NumPy, SciPy, pandas, matplotlib, scikit-learn and the optional
libraries listed further down). Minor version differences are harmless; a missing package is not,
and it is far better to find out now than in the middle of Notebook 10.

### 3.1 Versions of the core libraries

The helper below imports a package by name and returns its version, or `None` if it is not
installed. We will use the same trick for the optional libraries in Section 3.5.

In [ ]:
def version_of(name):
    """Return the version string of an importable package, or None if it is missing."""
    try:
        module = importlib.import_module(name)
    except Exception:            # ImportError, or a broken install that raises e.g. OSError
        return None
    return getattr(module, "__version__", "installed (no __version__)")

print(f"Python       {platform.python_version()}   ({platform.system()} {platform.release()}, {platform.machine()})")
print(f"executable   {sys.executable}")
print()
core = ["torch", "torchvision", "numpy", "scipy", "pandas", "matplotlib", "sklearn"]
for name in core:
    version = version_of(name)
    status = version if version is not None else "NOT INSTALLED  <-- install it from environment.yml"
    print(f"{name:<12} {status}")

If any of the core packages reports `NOT INSTALLED`, stop here and create the environment from
`environment.yml` (or `environment-cpu.yml` for a machine without an NVIDIA GPU, which creates an environment named `dl-course-cpu`):

```
conda env create -f environment.yml
conda activate dl-course
```

and then make sure the Jupyter kernel you selected is the one from that environment. The `executable`
line printed above tells you which Python interpreter the kernel is using; if the path does not
contain `dl-course`, you are running in a different environment, which is the single most common
cause of "but I installed it!" confusion.

### 3.2 CUDA and the GPU

The next cell reports whether PyTorch can see a CUDA GPU, which CUDA and cuDNN versions it was built
with, and how much memory each GPU has. `torch.cuda.mem_get_info` returns the memory that is free
*right now*, which matters if other programs (a browser with hardware acceleration, another
notebook) are already using the card.

In [ ]:
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA (build)   :", torch.version.cuda)
    print("cuDNN          :", torch.backends.cudnn.version())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}          : {props.name} | {props.total_memory / 1e9:.1f} GB | "
              f"compute capability {props.major}.{props.minor} | {props.multi_processor_count} SMs")
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    print(f"memory free now: {free_bytes / 1e9:.1f} GB of {total_bytes / 1e9:.1f} GB")
else:
    print("No CUDA GPU detected. The course runs on the CPU; set FAST_RUN = True for a first pass.")
    print("CPU threads used by PyTorch:", torch.get_num_threads())
mps_available = getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available()
print("Apple MPS backend available:", mps_available, "(the course uses cuda or cpu)")

If the first line says `True`, the GPU lines tell you what you have to work with: the compute
capability determines which features are available (bfloat16 and TF32 need 8.0 or higher, which
matters in Notebook 14 · Efficient Training and Scaling), and the free memory is what your training
runs can actually use. If it says `False` although the machine has an NVIDIA GPU, the environment
was probably created from the CPU build of PyTorch; Section 5.6 lists the fixes.

### 3.3 A timed matrix multiplication on CPU and GPU

Why do GPUs matter so much? Almost all of the arithmetic in a neural network is matrix
multiplication, and a matrix multiplication of two $n \times n$ matrices costs about $2n^3$
floating-point operations (each of the $n^2$ output entries is a dot product of length $n$, which is
$n$ multiplications and $n$ additions). A GPU has thousands of small arithmetic units that can work
on different output entries at the same time, so for large $n$ it can be one or two orders of
magnitude faster than a CPU.

The helper below times $a \cdot b$ for square matrices of side $n$. Two details are essential when
timing GPU code and are easy to get wrong. First, CUDA kernels are **asynchronous**: `a @ b` returns
immediately and the multiplication happens in the background, so we must call
`torch.cuda.synchronize()` before reading the clock. Second, the very first call on a device pays a
one-off cost (loading kernels, allocating memory), so we run one **warm-up** multiplication that we do
not time.

In [ ]:
def time_matmul(n, dev, repeats=3, dtype=torch.float32):
    """Average wall-clock seconds for one n x n @ n x n matrix multiplication on `dev`."""
    a = torch.randn(n, n, device=dev, dtype=dtype)
    b = torch.randn(n, n, device=dev, dtype=dtype)
    _ = a @ b                                     # warm-up (not timed)
    if dev.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(repeats):
        c = a @ b
    if dev.type == "cuda":
        torch.cuda.synchronize()                  # wait for the GPU to actually finish
    return (time.perf_counter() - t0) / repeats

sizes = [256, 512, 1024] if FAST_RUN else [256, 512, 1024, 2048]
devices_to_time = [torch.device("cpu")] + ([device] if device.type == "cuda" else [])

header = f"{'n':>6} | " + " | ".join(f"{d.type + ' seconds':>14}" for d in devices_to_time) + " | GFLOP/s (fastest)"
print(header); print("-" * len(header))
for n in sizes:
    seconds = [time_matmul(n, d) for d in devices_to_time]
    gflops = 2 * n ** 3 / min(seconds) / 1e9
    row = f"{n:>6} | " + " | ".join(f"{s:>14.5f}" for s in seconds) + f" | {gflops:>10.1f}"
    if len(seconds) == 2:
        row += f"   (GPU speed-up {seconds[0] / seconds[1]:.1f}x)"
    print(row)

Read the table from top to bottom. Doubling $n$ multiplies the work by eight, so the CPU time should
roughly increase eightfold per row (often a little less at the small end, because the fixed cost of
the call and the blocking strategy of the BLAS library make a larger multiplication more efficient
per operation). If you have a GPU, look at the speed-up column: for $n = 256$ it may be modest or even
below one, because launching a GPU kernel and synchronising costs tens of microseconds regardless of
the size of the problem, and a $256 \times 256$ multiplication is over before that overhead is paid
off. From $n = 1024$ upwards the speed-up should be substantial. This is the first lesson of GPU
computing: **GPUs are fast on large, regular workloads and slow on many tiny ones**, which is why we
process data in mini-batches rather than one example at a time.

The GFLOP/s number, billions of floating-point operations per second, is a useful yardstick. A laptop
CPU reaches tens to a few hundred GFLOP/s in float32; a modern GPU reaches tens of TFLOP/s (tens of
thousands of GFLOP/s), more still in the reduced-precision formats we meet in Notebook 14 ·
Efficient Training and Scaling. Exercise 5 at the end of this notebook explores how the speed-up
depends on the matrix size in more detail.

> **Common pitfall.** Timing GPU code without `torch.cuda.synchronize()` measures how long it takes to
> *launch* the work, not to *do* it, and produces absurdly optimistic numbers. Whenever you time
> anything on a GPU, synchronise before starting and before stopping the clock.

### 3.4 A matplotlib test plot

We have already drawn a timeline, so we know matplotlib works; this cell checks the two things that
most often go wrong with plotting in notebooks, namely mathematical text in labels (rendered by
matplotlib's built-in TeX subset) and multiple curves with a legend.

In [ ]:
xs = np.linspace(-2 * np.pi, 2 * np.pi, 400)
plt.plot(xs, np.sin(xs), label=r"$\sin(x)$")
plt.plot(xs, np.cos(xs), label=r"$\cos(x)$")
plt.plot(xs, np.tanh(xs), label=r"$\tanh(x)$ (an activation function)")
plt.xlabel("$x$"); plt.ylabel("$f(x)$"); plt.title("matplotlib is working")
plt.legend(); plt.tight_layout(); plt.show()

If the figure appears with properly typeset labels, plotting is ready. If nothing appears, your
notebook front end may need the inline backend enabled; in JupyterLab and VS Code this is the default.

### 3.5 Optional libraries

Later notebooks use a few larger libraries: Hugging Face `transformers`, `datasets` and `peft`
(Notebook 10), `gymnasium` (Notebook 13), `optuna` (Notebook 14), `onnx` and `onnxruntime`
(Notebook 15), and `torchinfo` and `tensorboard` (Notebook 05 onwards). Each of those notebooks
imports its library inside a `try/except` and degrades gracefully, but it is convenient to know now
which ones are present. Importing `transformers` can take a few seconds the first time.

In [ ]:
optional = {
    "transformers": "Notebook 10 · NLP with Pretrained Transformers (Hugging Face)",
    "datasets":     "Notebook 10 · NLP with Pretrained Transformers (Hugging Face)",
    "peft":         "Notebook 10 · NLP with Pretrained Transformers (Hugging Face)",
    "gymnasium":    "Notebook 13 · Reinforcement Learning",
    "optuna":       "Notebook 14 · Efficient Training and Scaling",
    "onnx":         "Notebook 15 · Deployment and Model Optimization",
    "onnxruntime":  "Notebook 15 · Deployment and Model Optimization",
    "torchinfo":    "Notebook 05 · The PyTorch Workflow and Best Practices",
    "tensorboard":  "Notebook 05 · The PyTorch Workflow and Best Practices",
}
missing = []
for name, used_in in optional.items():
    version = version_of(name)
    if version is None:
        missing.append(name)
        print(f"{name:<13} missing   -> needed by {used_in}")
    else:
        print(f"{name:<13} {version:<10} ok")
if missing:
    print(f"\n{len(missing)} optional package(s) missing: {', '.join(missing)}.")
    print("They are listed in environment.yml; the notebooks that need them will explain the fallback.")
else:
    print("\nAll optional packages are installed.")

A missing optional package is not an emergency. You can carry on with Parts I to III without any of
them, and install the rest before you reach the notebook that needs it. The point of running the
check now is that you will not be surprised later.

> **Key idea.** Verify the environment once, carefully, before starting; most "the code does not
> work" problems in a deep learning course are environment problems in disguise.

## 4. Deep learning in one page

Every deep learning system you will meet in this course, from the ten-line logistic regression of
Notebook 03 to the diffusion model of Notebook 12 · GANs and Diffusion Models, is built from the same
three ingredients and
trained by the same loop. Understanding this skeleton once means that every later notebook is a
variation on a theme rather than a new subject.

### 4.1 The three ingredients

**The model** is a function $f_\theta$ with adjustable parameters $\theta$ that maps an input $x$
to an output $\hat{y} = f_\theta(x)$. For a classifier the output is a score for each class; for a
regressor it is a number; for a language model it is a probability for each possible next token.
In deep learning the model is a neural network: a composition of layers, each of which is a simple
parametrised function such as an affine map $x \mapsto Wx + b$ followed by an element-wise
non-linearity such as $\mathrm{ReLU}(z) = \max(0, z)$. The parameters $\theta$ are the entries of
all the weight matrices $W$ and bias vectors $b$. The model defines *what the network can express*;
the parameters decide *which function in that family it currently is*.

**The loss** is a function $\mathcal{L}(\hat{y}, y)$ that measures how wrong a prediction $\hat{y}$ is
compared with the true target $y$, as a single non-negative number, with smaller meaning better.
Averaged over a dataset it becomes a function of the parameters alone,
$J(\theta) = \frac{1}{N}\sum_{i=1}^{N} \mathcal{L}(f_\theta(x_i), y_i)$, and training means finding
parameters that make $J$ small. The choice of loss encodes what we care about: mean-squared error for
regression, cross-entropy for classification. Notebook 01 · Mathematical Foundations derives both from
the principle of maximum likelihood, so that they will not look arbitrary.

**The optimizer** is the algorithm that adjusts $\theta$ to decrease $J$. Nearly always it is a
variant of **gradient descent**: compute the gradient $\nabla_\theta J$, the vector of partial
derivatives of the loss with respect to every parameter, and take a small step in the opposite
direction, $\theta \leftarrow \theta - \eta \nabla_\theta J$, where the **learning rate** $\eta$ sets
the step size. The gradient of a network with millions of parameters is computed by
**backpropagation**, and in PyTorch this is done for you by the **autograd** engine, the subject of
Notebook 03 · Tensors and Autograd. Modern optimizers such as Adam, used below, are gradient descent
with some extra bookkeeping to choose the step size per parameter; Notebook 04 · Training Deep
Networks: Optimization, Initialization, Normalization, Regularization derives them.

### 4.2 The training loop, in words

Picture the data flowing left to right and the gradients flowing right to left:

```
          forward pass  --------------------------------------------->
  inputs x  ==>  [ layer 1 ] ==> [ layer 2 ] ==> ... ==> outputs y_hat  ==>  loss(y_hat, y)
                     ^               ^                                            |
                     |   gradients   |                                            |
                     +---------------+--------------------------------------------+
          <---------------------------------------------  backward pass
                 optimizer.step(): every parameter moves a little against its gradient
```

One iteration of training consists of five steps, and the code below contains exactly one line for
each: (1) run the **forward pass** to obtain predictions; (2) evaluate the **loss**; (3) **zero** the
gradients left over from the previous iteration; (4) run the **backward pass** so that every
parameter's `.grad` holds $\partial J / \partial \theta$; (5) let the **optimizer** update the
parameters. Repeat for many iterations, usually on **mini-batches** (subsets of the data) rather than
the whole dataset; one pass through the whole dataset is called an **epoch**. That is the entire
algorithm. Everything else in the course is about choosing a good model, a good loss, a good
optimizer and good data, and about doing it fast.

### 4.3 A complete example in twenty-five lines

We now write it all down. The problem is a classic toy: scikit-learn's `make_moons` generates points
in the plane that form two interleaved half-moons, one per class, with some Gaussian noise. It is
two-dimensional so that we can draw everything, and it is not linearly separable, so a straight-line
classifier fails and a neural network has something to do. The model is a **multi-layer perceptron**
(MLP) with two hidden layers of 32 units, the loss is binary cross-entropy, the optimizer is Adam,
and we train on the full dataset in each step because it is tiny.

In [ ]:
from sklearn.datasets import make_moons

# 1. Data: 1000 points in the plane, two interleaved half-moons, labels 0/1
X_np, y_np = make_moons(n_samples=1000, noise=0.2, random_state=SEED)
X = torch.tensor(X_np, dtype=torch.float32, device=device)                 # shape (1000, 2)
y = torch.tensor(y_np, dtype=torch.float32, device=device).unsqueeze(1)    # shape (1000, 1)

# 2. Model: an MLP  2 -> 32 -> 32 -> 1  (the output is a logit, i.e. a pre-sigmoid score)
model = nn.Sequential(
    nn.Linear(2, 32), nn.ReLU(),
    nn.Linear(32, 32), nn.ReLU(),
    nn.Linear(32, 1),
).to(device)

# 3. Loss and optimizer
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

# 4. Training loop (full batch: all 1000 points in every step)
EPOCHS = 200 if FAST_RUN else 1000
model.train()
for epoch in range(1, EPOCHS + 1):
    logits = model(X)                       # (1) forward pass
    loss = loss_fn(logits, y)               # (2) how wrong are we?
    optimizer.zero_grad()                   # (3) clear the gradients of the previous step
    loss.backward()                         # (4) backpropagation: fills every parameter's .grad
    optimizer.step()                        # (5) move every parameter against its gradient
    if epoch % (EPOCHS // 5) == 0:
        train_acc = ((logits > 0).float() == y).float().mean().item()
        print(f"epoch {epoch:5d} | loss {loss.item():.4f} | train accuracy {train_acc:.3f}")

The loss falls steadily and the training accuracy climbs to the mid-to-high nineties. Some points
lie on the wrong side of the boundary because of the noise we added, so perfect accuracy is neither
attainable nor desirable: a model that classified every noisy training point correctly would have
memorised the noise, which is called **overfitting** and is a central theme of Notebook 04.

### 4.4 Looking at what the network learned

Because the input is two-dimensional we can evaluate the model on a fine grid covering the plane
and colour each point by the predicted probability of class 1. The black line is the **decision
boundary**, where the probability is exactly one half. The helper below will be reused in the
exercises, so it takes the model and the data as arguments and draws into an optional axis. Two
details deserve attention: `model.eval()` switches the model to evaluation mode (it makes no
difference for this model, but layers such as dropout and batch normalisation behave differently
during evaluation, so we build the habit now), and `torch.no_grad()` tells autograd not to record
operations, which saves memory and time whenever we are not going to call `backward()`.

In [ ]:
def plot_decision_boundary(model, X_np, y_np, title="Decision boundary", ax=None, resolution=300):
    """Colour the plane by the model's predicted P(class 1) and overlay the data points."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4.5))
    x_min, x_max = X_np[:, 0].min() - 0.5, X_np[:, 0].max() + 0.5
    y_min, y_max = X_np[:, 1].min() - 0.5, X_np[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution), np.linspace(y_min, y_max, resolution))
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32, device=device)
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(grid)).cpu().numpy().reshape(xx.shape)
    contours = ax.contourf(xx, yy, probs, levels=np.linspace(0, 1, 11), cmap="RdBu", alpha=0.75)
    ax.contour(xx, yy, probs, levels=[0.5], colors="k", linewidths=1)
    ax.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap="RdBu", edgecolors="k", s=15, linewidths=0.4)
    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$"); ax.set_title(title)
    return contours

contours = plot_decision_boundary(model, X_np, y_np, "MLP decision boundary on make_moons")
plt.colorbar(contours, label="predicted P(class 1)")
plt.tight_layout(); plt.show()

The boundary is a smooth curve that snakes between the two moons. It is not a straight line, which
is what a single linear layer could produce, and it is not a jagged shape that wraps around
individual noisy points either. Away from the data the colours become saturated: the network is
extremely confident about regions it has never seen, a characteristic of neural networks that we will
revisit when we discuss calibration and out-of-distribution detection in Notebook 16.

### 4.5 Every line explained

Let us go through the example line by line, with pointers to where each idea is treated in depth.

- `make_moons(n_samples=1000, noise=0.2, random_state=SEED)` returns a NumPy array `X_np` of shape
  `(1000, 2)` and a label vector `y_np` of shape `(1000,)` with entries 0 and 1. The `random_state`
  makes the dataset reproducible.
- `torch.tensor(X_np, dtype=torch.float32, device=device)` converts the NumPy array to a PyTorch
  **tensor**, the basic data structure of the library, in single precision (the default in deep
  learning; NumPy's default is double precision) and places it on the chosen device.
  `y.unsqueeze(1)` turns the shape `(1000,)` into `(1000, 1)` so that it matches the shape of the
  model's output. Shapes are the most frequent source of bugs in PyTorch code; Notebook 03 · Tensors
  and Autograd covers tensors, shapes and devices in detail.
- `nn.Sequential(nn.Linear(2, 32), nn.ReLU(), ...)` builds the model as a chain of **modules**.
  `nn.Linear(2, 32)` is an affine map from 2 inputs to 32 outputs; it owns a weight matrix of shape
  `(32, 2)` and a bias vector of shape `(32,)`, both initialised randomly. `nn.ReLU()` applies
  $\max(0, z)$ element-wise; without such non-linearities a stack of linear layers would collapse into
  a single linear layer (Notebook 02 proves this in one line). The final `nn.Linear(32, 1)` produces a
  single **logit** per point. `.to(device)` moves all the parameters to the GPU if we have one.
  Notebook 05 · The PyTorch Workflow and Best Practices dissects `nn.Module`.
- `nn.BCEWithLogitsLoss()` is binary cross-entropy applied to the sigmoid of the logit, computed in a
  numerically stable way. Notebook 01 derives cross-entropy from maximum likelihood and explains why
  the stable "with logits" form matters.
- `torch.optim.Adam(model.parameters(), lr=1e-2)` creates the optimizer and hands it every parameter
  tensor of the model. `lr` is the learning rate. Adam is the workhorse optimizer of the field;
  Notebook 04 derives it and compares it with plain SGD, momentum and others.
- `model.train()` puts the model in training mode. It matters for dropout and batch normalisation
  (Notebook 04); here it does nothing, but it is good hygiene.
- `logits = model(X)` is the **forward pass**: it calls the modules in order and returns a tensor of
  shape `(1000, 1)`. Behind the scenes autograd records every operation so that it can differentiate
  through them later.
- `loss = loss_fn(logits, y)` produces a single scalar tensor. Because it was computed from tensors
  that depend on the parameters, it "remembers" the whole computation.
- `optimizer.zero_grad()` clears the `.grad` attribute of every parameter. PyTorch **accumulates**
  gradients across `backward()` calls rather than overwriting them, which is useful for some advanced
  techniques (Notebook 14 uses it for gradient accumulation) but means we must zero them ourselves at
  every step. Forgetting this line is one of the classic bugs; Notebook 03 shows what goes wrong.
- `loss.backward()` runs **backpropagation**: autograd walks the recorded computation backwards and
  fills `p.grad` with $\partial \text{loss} / \partial p$ for every parameter `p`. Notebook 02 derives
  this by hand; Notebook 03 builds a miniature autograd engine so that it is not magic.
- `optimizer.step()` updates every parameter using its gradient, following Adam's rules.
- The `if` block prints progress. `(logits > 0)` is the predicted class (a logit above zero means a
  probability above one half), and comparing it with `y` gives the training accuracy. `.item()`
  converts a one-element tensor to a Python number.

Two things are missing from this example that a real project would have: a **held-out test set** to
measure generalisation, and **mini-batches** with a `DataLoader`. We add the first one now because it
is essential to interpreting any result; the second is unnecessary for a thousand points and is
introduced properly in Notebook 05.

### 4.6 A reusable training helper and a proper test set

The exercises will train this kind of model many times with different widths, learning rates and
datasets, so we package the recipe into two functions: `train_mlp` builds and trains a model
(re-seeding first, so that runs are comparable) and returns it together with the loss history, and
`accuracy` evaluates a model on any dataset without recording gradients. We also generate a fresh
test set from the same distribution with a different random seed. The gap between training and test
accuracy is our first measurement of generalisation.

In [ ]:
def make_mlp(hidden=32, depth=2):
    """An MLP 2 -> hidden -> ... -> hidden -> 1 with `depth` hidden layers and ReLU activations."""
    layers, d_in = [], 2
    for _ in range(depth):
        layers += [nn.Linear(d_in, hidden), nn.ReLU()]
        d_in = hidden
    layers.append(nn.Linear(d_in, 1))
    return nn.Sequential(*layers).to(device)


def train_mlp(X, y, hidden=32, lr=1e-2, epochs=None, seed=SEED):
    """Train a fresh MLP on (X, y) with Adam and full-batch steps. Returns (model, losses)."""
    if epochs is None:
        epochs = 200 if FAST_RUN else 1000
    torch.manual_seed(seed)                        # same initial weights for every configuration
    model = make_mlp(hidden)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    losses = []
    model.train()
    for _ in range(epochs):
        loss = loss_fn(model(X), y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        losses.append(loss.item())
    return model, losses


@torch.no_grad()
def accuracy(model, X, y):
    """Fraction of correctly classified points (logit > 0 means class 1)."""
    model.eval()
    return ((model(X) > 0).float() == y).float().mean().item()


X_test_np, y_test_np = make_moons(n_samples=1000, noise=0.2, random_state=SEED + 1)
X_test = torch.tensor(X_test_np, dtype=torch.float32, device=device)
y_test = torch.tensor(y_test_np, dtype=torch.float32, device=device).unsqueeze(1)

model, losses = train_mlp(X, y)
print(f"train accuracy {accuracy(model, X, y):.3f} | test accuracy {accuracy(model, X_test, y_test):.3f}")
plt.plot(losses)
plt.xlabel("epoch"); plt.ylabel("training loss (binary cross-entropy)")
plt.title("Loss curve of the tiny MLP"); plt.tight_layout(); plt.show()

The loss curve drops quickly during the first few dozen epochs and then flattens; this shape, fast
initial progress followed by slow refinement, is what almost every healthy training run looks like.
The test accuracy is close to the training accuracy, which tells us the model has learned the
shape of the moons rather than the particular noisy points. With noise 0.2 the two classes overlap
slightly, so the best achievable accuracy on this problem is a little below 100%; a model that
reported 100% on the training set would be a warning sign, not a success.

> **Try it.** Change `noise=0.2` to `noise=0.4` in the data cell and re-run the cells of this section.
> The boundary becomes more contorted and the gap between training and test accuracy opens up.
> Exercise 4 studies this systematically.

> **Key idea.** Model, loss, optimizer, and a loop of forward pass, backward pass and update:
> every deep learning system in this course has exactly this structure.

## 5. How to study this course

The notebooks are designed to be *run*, not read. Deep learning is an empirical discipline: the
theory tells you what should happen, and the experiment tells you what does, and the two do not
always agree. Here is the way of working that we have found most effective for self-paced learners.

### 5.1 Active reading

Before running a code cell, read the markdown cell above it and predict what the output will be: the
shape of a tensor, the approximate value of a loss, the form of a plot. Then run the cell and compare.
When your prediction was wrong, that is the moment to slow down; it means the model in your head
differs from the one in the computer, and reconciling them is where the learning happens. After the
cell, read the interpretation and check that you could have written it yourself.

### 5.2 Change the hyperparameters

Every experiment in the course depends on a handful of numbers: learning rate, hidden width, number
of epochs, batch size, amount of noise, regularisation strength. These are called
**hyperparameters** to distinguish them from the parameters the optimizer learns. Change them. Halve
the learning rate and double it; make the network absurdly small and absurdly large; train for
one epoch and for fifty. The `> **Try it.**` callouts suggest specific changes, but do not limit
yourself to them. Intuition about how these knobs interact is the most transferable skill a deep
learning practitioner has, and it cannot be acquired by reading.

### 5.3 Keep an experiment log

When you change things, write down what you changed and what happened, even for the small
experiments in this course. Memory is unreliable and results are easy to misattribute. A log can be
a text file, a spreadsheet, or a list of dictionaries in the notebook that you print as a table; the
form does not matter, the habit does. Notebook 05 introduces TensorBoard and configuration files for
serious tracking, and Notebook 17 · Capstone Projects discusses reporting. The cell below shows the
simplest possible log: a list of dictionaries, one per run, displayed with pandas (imported in the
setup cell as `pd`).

In [ ]:
experiment_log = []

def log_run(**fields):
    """Append one experiment record (any keyword fields) with a timestamp."""
    fields["time"] = time.strftime("%H:%M:%S")
    experiment_log.append(fields)

for hidden in [4, 32]:
    for lr in [1e-3, 1e-2]:
        trained, hist = train_mlp(X, y, hidden=hidden, lr=lr, epochs=100 if FAST_RUN else 300)
        log_run(hidden=hidden, lr=lr, epochs=len(hist), final_loss=round(hist[-1], 4),
                train_acc=round(accuracy(trained, X, y), 3), test_acc=round(accuracy(trained, X_test, y_test), 3))

pd.DataFrame(experiment_log)

Four runs, four rows, and already a small story: the wider network reaches a lower loss, and the
larger learning rate gets there faster within the fixed budget. Each row is a fact you no longer have
to remember. When an experiment matters, add the git commit, the random seed and the software
versions to the record too.

### 5.4 How to read error messages

PyTorch error messages are long but informative, and learning to read them is a skill worth
practising on purpose. The cell below deliberately feeds the model an input with the wrong number of
features and catches the exception so that we can inspect it.

In [ ]:
try:
    wrong_input = torch.randn(8, 3, device=device)     # 8 points with 3 features...
    model(wrong_input)                                  # ...but the model expects 2 features
except RuntimeError as err:
    print("RuntimeError:", err)

The message says that "mat1 and mat2 shapes cannot be multiplied" and gives the two shapes:
`8x3` is our input, `2x32` is the transposed weight of the first `nn.Linear(2, 32)`. The inner
dimensions (3 and 2) do not match. This tells you, without any guesswork, which layer failed and
what it expected. A general strategy for any traceback:

1. **Read the last line first.** It names the exception and usually contains the decisive shape,
   dtype or device information.
2. **Find the first line of the traceback that is in *your* code**, not in PyTorch's internals. That
   is where the wrong tensor was created or passed.
3. **Print shapes** (`print(x.shape, x.dtype, x.device)`) just before the failing line. Shape,
   dtype and device mismatches account for the large majority of errors.
4. **Search for the last line verbatim.** Nearly every error message has been asked about publicly,
   and the first answer is usually right.

Notebook 05 has a dedicated section on debugging, including hooks, anomaly detection and a checklist.

### 5.5 When to use a GPU

The first half of the course, up to Notebook 05, runs comfortably on a CPU even with `FAST_RUN =
False`. From Notebook 06 · Convolutional Neural Networks onwards, the full experiments train
convolutional and Transformer models on tens of thousands of images or characters, and a GPU turns
twenty minutes into two. If you do not have one, use `FAST_RUN = True` on the CPU for a first pass
and run the full experiments on a hosted notebook service.

### 5.6 Common environment problems and their fixes

- **"CUDA not available" although you have an NVIDIA GPU.** Either the PyTorch build is the CPU-only
  one (check `torch.version.cuda`; it is `None` for CPU builds), or the NVIDIA driver is older than
  the CUDA version PyTorch was built with. Recreate the environment from `environment.yml`, which pins
  a CUDA build, and update the driver. Note that PyTorch's pip wheels bundle their own CUDA runtime,
  so you do not need to install the CUDA toolkit separately.
- **`torch.cuda.OutOfMemoryError`.** The batch is too large for the card, or a previous cell left
  large tensors alive. Halve the batch size; delete tensors you no longer need and call
  `torch.cuda.empty_cache()`; restart the kernel if a crashed cell left memory allocated. Notebook 14
  shows how to estimate memory before running.
- **DataLoader hangs or crashes on Windows.** With `num_workers > 0`, DataLoader spawns worker
  processes, which on Windows requires the code to be importable and can fail inside notebooks.
  The course uses `num_workers=0` everywhere; if you increase it, wrap your code in
  `if __name__ == "__main__":` when running as a script.
- **The kernel dies or restarts.** Usually the process ran out of system RAM (loading a full dataset
  as float64 NumPy arrays, for example), or a native library crashed. Watch memory use, load data in
  float32, and reduce `N_TRAIN` under `FAST_RUN`.
- **Results differ between runs.** Some GPU kernels are non-deterministic for speed. Seeding gives
  reproducibility on the CPU and near-reproducibility on the GPU; Notebook 05 explains
  `torch.use_deterministic_algorithms` and its costs.
- **Stale `.pyc` files, mixed environments, `ModuleNotFoundError` despite installation.** Check the
  interpreter path printed in Section 3.1; the kernel is almost certainly not the environment you
  installed into.

## 6. Notation and conventions

The course uses a consistent notation for mathematics and a consistent set of shape conventions for
code. Both are summarised here for reference; you do not need to memorise the table now, but you will
want to come back to it.

### 6.1 Mathematical notation

| Symbol | Meaning | Example |
|--------|---------|---------|
| $a, b, \alpha, \eta$ | scalars (lower-case italic; Greek for hyperparameters) | learning rate $\eta$ |
| $\mathbf{x}, \mathbf{w}, \mathbf{h}$ | vectors (bold lower-case), always column vectors | input $\mathbf{x} \in \mathbb{R}^d$ |
| $\mathbf{W}, \mathbf{X}, \mathbf{A}$ | matrices (bold upper-case) | weight matrix $\mathbf{W} \in \mathbb{R}^{m \times n}$ |
| $\mathsf{X}$ | tensors of rank three or more | image batch $\mathsf{X} \in \mathbb{R}^{N \times C \times H \times W}$ |
| $x_i$, $W_{ij}$ | $i$-th entry of a vector, entry $(i, j)$ of a matrix | |
| $\mathbf{x}^{(i)}$ | the $i$-th example in a dataset | $\{(\mathbf{x}^{(i)}, y^{(i)})\}_{i=1}^{N}$ |
| $N$, $d$, $B$ | dataset size, input dimension, batch size | |
| $\theta$ | all parameters of a model, stacked into one vector | $f_\theta$ |
| $\hat{y}$ | a prediction (hat = estimated) | $\hat{y} = f_\theta(\mathbf{x})$ |
| $\mathcal{L}$, $J(\theta)$ | per-example loss, average loss over the data | |
| $\nabla_\theta J$ | gradient of $J$ with respect to $\theta$ | $\theta \leftarrow \theta - \eta \nabla_\theta J$ |
| $\frac{\partial \mathbf{y}}{\partial \mathbf{x}}$ | Jacobian matrix, shape (dim $\mathbf{y}$) $\times$ (dim $\mathbf{x}$) | Notebook 01 |
| $\sigma(\cdot)$, $\mathrm{softmax}(\cdot)$ | the logistic sigmoid; the softmax function | |
| $\mathbb{E}[\cdot]$, $\mathrm{Var}[\cdot]$ | expectation, variance | |
| $p(y \mid \mathbf{x})$ | conditional probability (or density) of $y$ given $\mathbf{x}$ | |
| $\mathcal{N}(\mu, \sigma^2)$ | Gaussian distribution with mean $\mu$ and variance $\sigma^2$ | |
| $\mathbf{1}$, $\mathbf{I}$ | vector of ones; identity matrix | |
| $\lVert \mathbf{x} \rVert_2$, $\lVert \mathbf{W} \rVert_F$ | Euclidean norm; Frobenius norm | weight decay $\lambda \lVert \theta \rVert_2^2$ |
| $\odot$ | element-wise (Hadamard) product | |
| $\log$ | natural logarithm, always | cross-entropy $-\log p(y \mid \mathbf{x})$ |

Two conventions are worth stating explicitly because textbooks differ. A **linear layer** is written
$\mathbf{h} = \mathbf{W}\mathbf{x} + \mathbf{b}$ in mathematics, with $\mathbf{W} \in \mathbb{R}^{m \times n}$
mapping $n$ inputs to $m$ outputs, but in code a batch of inputs is a matrix $\mathbf{X}$ with one
example *per row*, so the same layer is computed as $\mathbf{X}\mathbf{W}^\top + \mathbf{b}$, which is
exactly what `nn.Linear` does (its `weight` attribute has shape `(out_features, in_features)`).
And **gradients have the same shape as the thing they are taken with respect to**: the gradient of a
scalar loss with respect to a $(m \times n)$ weight matrix is itself $(m \times n)$, which is what
makes the update $\mathbf{W} \leftarrow \mathbf{W} - \eta \, \partial J / \partial \mathbf{W}$ well
defined.

### 6.2 Shape conventions in code

Shapes are written as tuples with named axes in comments. The first axis is almost always the batch.

| Kind of data | Shape | Notes |
|--------------|-------|-------|
| tabular / feature vectors | `(batch, features)` | `nn.Linear` acts on the last axis |
| images | `(batch, channels, height, width)` | PyTorch is channels-first; matplotlib wants `(height, width, channels)` |
| sequences | `(batch, time, features)` | we use `batch_first=True` for RNNs and Transformers |
| token ids | `(batch, time)` | integer dtype `torch.long` |
| class labels | `(batch,)` | integer class indices, not one-hot |
| logits of a classifier | `(batch, classes)` | cross-entropy takes logits, not probabilities |
| a scalar loss | `()` | zero-dimensional tensor; `.item()` gives the Python number |

The cell below creates a tensor of each kind and prints its shape, the number of axes and the number
of elements, and shows the batched linear layer computation as a matrix product.

In [ ]:
examples = {
    "tabular":  torch.zeros(64, 784),             # (batch, features)
    "images":   torch.zeros(64, 3, 32, 32),       # (batch, channels, height, width)
    "sequence": torch.zeros(64, 50, 128),         # (batch, time, features)
    "tokens":   torch.zeros(64, 50, dtype=torch.long),
    "labels":   torch.zeros(64, dtype=torch.long),
    "loss":     torch.tensor(0.6931),             # a scalar
}
for name, t in examples.items():
    print(f"{name:<9} shape={str(tuple(t.shape)):<18} ndim={t.ndim}  numel={t.numel():>8,}  dtype={t.dtype}")

# The batched linear layer: X (batch, in) @ W^T (in, out) + b (out,) -> (batch, out)
layer = nn.Linear(784, 256)
X_tab = torch.randn(64, 784)                      # random inputs, so the equality check is not trivial
manual = X_tab @ layer.weight.T + layer.bias
print("\nnn.Linear weight shape:", tuple(layer.weight.shape), "| output shape:", tuple(layer(X_tab).shape),
      "| matches manual X @ W^T + b:", torch.allclose(manual, layer(X_tab)))

Note the two tensors that are *not* floating point: token ids and class labels are `torch.long`
(64-bit integers), because they index into embedding tables and loss functions. Passing a float
label vector of shape `(batch,)` to `nn.CrossEntropyLoss` is another classic error, and its message
(on the CPU, "expected scalar type Long but found Float") makes sense once you know this convention.

> **Key idea.** Batch first, channels before spatial dimensions, integer labels, logits into the loss:
> if a shape surprises you, come back to this table.

## Exercises

The exercises use the data (`X`, `y`, `X_test`, `y_test`), the helpers (`train_mlp`, `accuracy`,
`plot_decision_boundary`, `time_matmul`) and the `model` defined above. Write your own attempt in a
new cell before opening the solution.

### Exercise 1 (warm-up): counting parameters

The model `nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))`
has a certain number of trainable parameters. (a) Compute the number by hand, remembering that each
`nn.Linear(n_in, n_out)` has an `n_out × n_in` weight matrix and an `n_out` bias vector.
(b) Write a function `count_parameters(model)` that returns the number of trainable parameters of any
PyTorch module, and check it against your hand computation. (c) Print the name, shape and number of
elements of every parameter tensor using `model.named_parameters()`.

**Solution to Exercise 1**

By hand: the first layer has $32 \times 2 = 64$ weights and $32$ biases, the second has
$32 \times 32 = 1024$ weights and $32$ biases, and the output layer has $1 \times 32 = 32$ weights and
$1$ bias, so in total $64 + 32 + 1024 + 32 + 32 + 1 = 1185$ parameters. In code, every parameter
tensor `p` reports its element count through `p.numel()`, and `p.requires_grad` tells us whether it
is trainable (all of them are here; in Notebook 07 we will freeze some layers, and then the
distinction matters). The general lesson is that the hidden-to-hidden layer dominates the count:
parameters grow with the *product* of the widths, which is why wide networks get large quickly.

In [ ]:
def count_parameters(module, trainable_only=True):
    """Total number of (trainable) parameters of a module."""
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)

by_hand = (32 * 2 + 32) + (32 * 32 + 32) + (1 * 32 + 1)
print(f"by hand: {by_hand} | count_parameters: {count_parameters(model)} | agree: {by_hand == count_parameters(model)}\n")
for name, p in model.named_parameters():
    print(f"{name:<10} shape={str(tuple(p.shape)):<10} numel={p.numel():>5}  requires_grad={p.requires_grad}")

The parameter names (`0.weight`, `0.bias`, `2.weight`, ...) are the indices of the modules inside the
`nn.Sequential` followed by the attribute name; the ReLU modules at indices 1 and 3 have no
parameters and therefore do not appear. Notebook 05 shows how to give layers meaningful names.

### Exercise 2: the effect of model width

Train the MLP with hidden widths 1, 2, 4, 16, 64 and 256 (keep the depth at two hidden layers and
the learning rate at `1e-2`), using `train_mlp`. For each width report the training and test
accuracy and draw the decision boundary in a 2×3 grid of subplots. Which widths underfit? Does the
largest width overfit?

**Solution to Exercise 2**

`train_mlp` re-seeds before building each model, so the only thing that changes between runs is the
width. With one or two hidden units per layer the network can only carve the plane with one or two
ReLU "corners", which is not enough to follow two interleaved crescents, and it underfits at
roughly 88% accuracy, about what a straight cut through the moons achieves; width 4 is on the edge
and, depending on the budget, may or may not find the curved solution. From width 16 upwards the
boundary is essentially the same smooth curve and the test accuracy stops improving at about 97%.
The largest width does not overfit noticeably here: with up to about 67 000 parameters but a
smooth two-dimensional problem, Adam finds a well-behaved solution long before it starts memorising
noise. Over-parametrised networks that generalise well are a recurring theme of modern deep
learning, and Notebook 04 examines when they do and do not.

In [ ]:
widths = [1, 2, 4, 16, 64, 256]
fig, axes = plt.subplots(2, 3, figsize=(13, 7.5))
for width, ax in zip(widths, axes.ravel()):
    trained, hist = train_mlp(X, y, hidden=width)
    tr, te = accuracy(trained, X, y), accuracy(trained, X_test, y_test)
    print(f"width {width:>4} | params {count_parameters(trained):>6} | final loss {hist[-1]:.4f} | "
          f"train acc {tr:.3f} | test acc {te:.3f}")
    plot_decision_boundary(trained, X_np, y_np, f"width {width}: train {tr:.2f} / test {te:.2f}", ax=ax, resolution=150)
plt.tight_layout(); plt.show()

Reading the grid left to right and top to bottom, the boundary evolves from a single straight cut
(width 1) to a curve that follows the moons (width 16 and beyond). The numbers confirm the picture:
the test accuracy jumps between widths 1 and 4 and then plateaus. Capacity that the problem does not
need is, in this case, simply unused.

### Exercise 3: the effect of the learning rate

Train the width-32 MLP with learning rates $10^{-4}, 10^{-3}, 10^{-2}, 10^{-1}$ and $1$, and plot the
five loss curves on one figure with a logarithmic vertical axis. Describe the four regimes you see
(too small, good, too large, divergent) and report the final test accuracy of each run.

**Solution to Exercise 3**

The learning rate is the single most important hyperparameter, and this experiment shows why. At
$10^{-4}$ Adam takes steps so small that the loss is still falling slowly when the budget runs out:
the model *would* get there, but not in time. At $10^{-3}$ and $10^{-2}$ the loss falls quickly and
smoothly; $10^{-2}$ is the sweet spot for this problem. At $10^{-1}$ the curve becomes jagged: each
step overshoots the valley floor and the optimizer bounces from side to side, and although the loss
still decreases on average it never settles. At $1$ the updates are so large that the network is
thrown into regions where the loss is far worse than at initialisation; with Adam the loss stays
finite (its per-parameter normalisation caps the step size), but with plain gradient descent it
would explode to infinity. Notebook 04 turns this qualitative picture into a quantitative rule
involving the curvature of the loss, and introduces learning-rate schedules and range tests for
choosing $\eta$ systematically.

In [ ]:
learning_rates = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]
plt.figure(figsize=(8, 4.5))
for lr in learning_rates:
    trained, hist = train_mlp(X, y, hidden=32, lr=lr)
    plt.plot(hist, label=f"lr = {lr:g}  (test acc {accuracy(trained, X_test, y_test):.3f})")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("training loss (log scale)")
plt.title("Learning-rate sweep for the tiny MLP"); plt.legend(); plt.tight_layout(); plt.show()

Notice how much the *shape* of a loss curve tells you before you look at any accuracy number: a
flat-but-decreasing line means "increase the learning rate", a jagged line means "decrease it", and
a curve that goes up means "decrease it a lot". Learning to read loss curves is a skill that Notebook
04 develops into a diagnostic checklist.

### Exercise 4: the effect of dataset noise

Generate training and test sets with `make_moons` for noise levels 0.0, 0.1, 0.2, 0.3, 0.4 and 0.5
(1000 points each, different seeds for train and test; the noise is Gaussian noise added to the
coordinates, not to the labels), train the width-32 MLP on each, and plot the training and test
accuracy as a function of the noise level. Explain the gap between the two curves
and why even the test accuracy must decrease as the noise grows.

**Solution to Exercise 4**

Two effects appear as the noise grows. First, the two moons start to overlap, so that points of both
classes occupy the same regions of the plane; no classifier, however good, can separate them there,
and the best achievable test accuracy falls (its complement, the smallest possible error rate, is
called the **Bayes error** in the theory). Second,
the network has enough capacity to bend its boundary around individual noisy training points, which
raises the training accuracy above what generalises: that is the widening gap between the two curves,
the signature of overfitting. At zero noise the moons are perfectly separable and both accuracies
reach essentially 100% (with `noise=0.0`, `make_moons` places the points deterministically on the
two arcs, so the training and test sets even coincide). The gap is the quantity that regularisation (weight decay, dropout, early stopping, data
augmentation) tries to close, and Notebook 04 devotes a full section to it.

In [ ]:
noise_levels = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
train_accs, test_accs = [], []
for noise in noise_levels:
    Xtr_np, ytr_np = make_moons(n_samples=1000, noise=noise, random_state=SEED)
    Xte_np, yte_np = make_moons(n_samples=1000, noise=noise, random_state=SEED + 1)
    Xtr = torch.tensor(Xtr_np, dtype=torch.float32, device=device)
    ytr = torch.tensor(ytr_np, dtype=torch.float32, device=device).unsqueeze(1)
    Xte = torch.tensor(Xte_np, dtype=torch.float32, device=device)
    yte = torch.tensor(yte_np, dtype=torch.float32, device=device).unsqueeze(1)
    trained, _ = train_mlp(Xtr, ytr, hidden=32)
    train_accs.append(accuracy(trained, Xtr, ytr)); test_accs.append(accuracy(trained, Xte, yte))
    print(f"noise {noise:.1f} | train acc {train_accs[-1]:.3f} | test acc {test_accs[-1]:.3f} | gap {train_accs[-1] - test_accs[-1]:+.3f}")

plt.plot(noise_levels, train_accs, "o-", label="train accuracy")
plt.plot(noise_levels, test_accs, "s-", label="test accuracy")
plt.xlabel("noise level of make_moons"); plt.ylabel("accuracy"); plt.ylim(0.5, 1.02)
plt.title("Generalisation gap versus dataset noise"); plt.legend(); plt.tight_layout(); plt.show()

Both curves fall with the noise. How far the gap opens depends on the training budget: memorising
individual noisy points takes many epochs, so under `FAST_RUN` (200 epochs) the two curves stay
close and the gap may even fluctuate around zero, while with the full budget of 1000 epochs, and
still more with a wider network or more epochs, the training accuracy pulls ahead of the test
accuracy at the higher noise levels. The Bayes-error effect, both curves falling as the classes
overlap, is robust to all of these choices. A useful mental habit: whenever you see a training
accuracy, ask what the test accuracy is, and whenever you see a test accuracy, ask what the best
possible accuracy on that data could be.

### Exercise 5 (challenge): GPU versus CPU across matrix sizes

Using `time_matmul`, measure the time of a square matrix multiplication for sizes
$n = 64, 128, 256, 512, 1024, 2048$ (stop at 1024 under `FAST_RUN`) on the CPU and, if available, on
the GPU, and plot time against $n$ on log–log axes together with the achieved GFLOP/s. Where is the
crossover below which the GPU is not worth using? If you have no GPU, compare float32 with float64
on the CPU instead and explain the difference you measure. The solution must run on either kind of
machine.

**Solution to Exercise 5**

The solution builds a list of `(label, device, dtype)` configurations that depends on the hardware:
CPU float32 and CPU float64 always, plus GPU float32 when CUDA is present. On a log–log plot the CPU
curve for large $n$ is a straight line of slope about 3 (time proportional to $n^3$), and the
achieved GFLOP/s rises with $n$ and then saturates at the throughput of the hardware. The GPU curve
is nearly *flat* for small $n$, because the time is dominated by the fixed cost of launching the
kernel and synchronising, and only starts rising with $n^3$ once the work exceeds that overhead; the
crossover with the CPU is typically somewhere between $n = 128$ and $n = 512$. On the CPU, float64
takes about twice as long as float32 for large matrices, because each number occupies twice the
memory bandwidth and the vector units process half as many per instruction; this is the first reason
deep learning uses float32 (and increasingly float16 and bfloat16, see Notebook 14) rather than
NumPy's default float64.

In [ ]:
sizes_ex = [64, 128, 256, 512, 1024] if FAST_RUN else [64, 128, 256, 512, 1024, 2048]
configs = [("CPU float32", torch.device("cpu"), torch.float32), ("CPU float64", torch.device("cpu"), torch.float64)]
if device.type == "cuda":
    configs.append(("GPU float32", device, torch.float32))

results = {label: [] for label, _, _ in configs}
for n in sizes_ex:
    for label, dev, dtype in configs:
        results[label].append(time_matmul(n, dev, repeats=3, dtype=dtype))

fig, (ax_time, ax_flops) = plt.subplots(1, 2, figsize=(12, 4.2))
for label, seconds in results.items():
    ax_time.loglog(sizes_ex, seconds, "o-", label=label)
    ax_flops.semilogx(sizes_ex, [2 * n ** 3 / s / 1e9 for n, s in zip(sizes_ex, seconds)], "o-", label=label)
ax_time.set_xlabel("matrix side n"); ax_time.set_ylabel("seconds per matmul"); ax_time.set_title("Time (log-log)")
ax_flops.set_xlabel("matrix side n"); ax_flops.set_ylabel("GFLOP/s"); ax_flops.set_title("Achieved throughput")
ax_time.legend(); ax_flops.legend(); plt.tight_layout(); plt.show()

print(f"{'n':>6}" + "".join(f"{label:>16}" for label in results))
for i, n in enumerate(sizes_ex):
    print(f"{n:>6}" + "".join(f"{results[label][i]:>16.6f}" for label in results))
if device.type == "cuda":
    speedups = [c / g for c, g in zip(results["CPU float32"], results["GPU float32"])]
    crossover = next((n for n, s in zip(sizes_ex, speedups) if s > 1), None)
    print("\nGPU speed-up over CPU float32 by size:", ", ".join(f"n={n}: {s:.1f}x" for n, s in zip(sizes_ex, speedups)))
    print("first size where the GPU wins:", crossover)
else:
    ratios = [d / s for s, d in zip(results["CPU float32"], results["CPU float64"])]
    print("\nfloat64 / float32 time ratio by size:", ", ".join(f"n={n}: {r:.2f}x" for n, r in zip(sizes_ex, ratios)))

The throughput panel is the one to remember. Small matrices, whatever the device, use a small
fraction of the hardware's capacity; the cure is to batch work together so that each operation is
large. This is the reason mini-batches exist, the reason that "fusing" small operations into one is
a major optimisation (Notebook 14 · Efficient Training and Scaling), and the reason that per-example
Python loops over tensors are so slow that Notebook 01 devotes a section to vectorisation.

## Summary

- Deep learning is machine learning with learned representations: a stack of simple differentiable
  layers trained end-to-end by gradient descent. It excels on large, structured data (images, text,
  audio, sequences) and is a poor choice for small tabular problems, causal questions, or decisions
  that need guarantees.
- The field's history is a sequence of ideas (perceptron, backpropagation, convolutions, recurrence,
  ReLU and dropout, residual connections, attention) that became decisive only when data and compute
  caught up, from AlexNet in 2012 to the Transformer in 2017 and the scaled-up language and diffusion
  models of the 2020s.
- The course has eighteen notebooks in seven parts, from mathematics and from-scratch networks through
  training, vision, language, generative models and reinforcement learning to engineering, trust and
  projects. Every notebook has worked exercises and a `FAST_RUN` switch for a quick CPU pass.
- Every deep learning system consists of a **model** $f_\theta$, a **loss** $\mathcal{L}$ and an
  **optimizer**, trained by the loop *forward pass, loss, zero gradients, backward pass, update*.
- The complete example in Section 4 — `make_moons`, an MLP built with `nn.Sequential`,
  `BCEWithLogitsLoss`, `Adam` and a twenty-five-line loop — reaches over 95% accuracy, and its
  decision-boundary plot shows what the network learned.
- Model width, learning rate and input noise each change the outcome in characteristic ways: too
  little capacity underfits, a learning rate that is too large oscillates or diverges, and noise opens
  a gap between training and test accuracy (overfitting).
- GPUs are fast on large, regular workloads and slow on many tiny ones; always synchronise before
  timing GPU code; batch work to use the hardware.
- Most "the code does not work" problems are environment problems: check the interpreter path, the
  CUDA build, memory, and use `num_workers=0` in notebooks on Windows.
- Shape conventions: batch first, `(batch, channels, height, width)` for images,
  `(batch, time, features)` for sequences, integer labels, logits into the loss.

## Further reading

- Goodfellow, I., Bengio, Y. and Courville, A. (2016). *Deep Learning*. MIT Press. Chapter 1 gives
  a longer version of the history and the representation-learning perspective of Section 1.
- LeCun, Y., Bengio, Y. and Hinton, G. (2015). Deep learning. *Nature*, 521, 436–444. A short,
  readable survey by three founders of the field, written just before the Transformer.
- Zhang, A., Lipton, Z. C., Li, M. and Smola, A. J. (2023). *Dive into Deep Learning*. Cambridge
  University Press; free online at https://d2l.ai. A code-first textbook whose PyTorch examples
  parallel much of this course.
- Prince, S. J. D. (2023). *Understanding Deep Learning*. MIT Press. Free online; excellent figures
  and a modern selection of topics.
- Bishop, C. M. and Bishop, H. (2024). *Deep Learning: Foundations and Concepts*. Springer. A
  mathematically careful treatment, good companion for Notebooks 01 and 04.
- Krizhevsky, A., Sutskever, I. and Hinton, G. E. (2012). ImageNet Classification with Deep
  Convolutional Neural Networks. *NeurIPS*. The paper that started the modern era.
- Rumelhart, D. E., Hinton, G. E. and Williams, R. J. (1986). Learning representations by
  back-propagating errors. *Nature*, 323, 533–536.
- Vaswani, A. et al. (2017). Attention Is All You Need. *NeurIPS*.
- Karpathy, A. (2019). *A Recipe for Training Neural Networks* (blog post). Practical advice on the
  empirical workflow described in Section 5.
- The PyTorch documentation, https://docs.pytorch.org/docs/stable/, in particular the "Learn the
  Basics" tutorial series, which covers the same ground as Section 4 at slightly greater length.